Groundwater | Transport Modeling Track

# Step 5: Calibration

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In [04t_model_implementation.ipynb](04t_model_implementation.ipynb) you built the coupled flow-and-transport spill→capture model, met the grid–Péclet problem, and worked with the locked transport parameters ($\alpha_L = 10$ m, $\alpha_T = 1$ m, $\phi_e = 0.20$).

**Why this notebook is short.** A *full* transport calibration — estimating $\alpha_L$, $\alpha_T$ and $\phi_e$ from observed plume data — is **out of scope here**: in the field you rarely have enough concentration observations to constrain those parameters, and they trade off against one another. Calibration rigour lives in the **flow** phase (heads, fluxes, PEST++). So Step 5 is deliberately a **short bridge** — it explains *where* calibration would fit, restates the defensible-claim idea, and offers one optional dispersivity demonstration.

| | |
|---|---|
| **Read time** | ~10–15 min |
| **Optional α_L demo** | +5 min (light analytical calc) |

## What this bridge covers

This notebook does **not** calibrate anything. Instead it:

1. explains **why transport calibration is out of scope** and where the rigour actually comes from;
2. restates the **defensible-claim idea** — which results a coarse-transverse grid supports, and which it does not;
3. offers **one optional, illustrative** $\alpha_L$ demonstration — predict-then-see how dispersivity reshapes a breakthrough curve.

> **Conservative-solute framing.** Throughout the demos we model a conservative tracer / contaminant (no sorption, no decay) with **locked** parameters: $\alpha_L = 10$ m, $\alpha_T = 1$ m, $\phi_e = 0.20$. Rather than re-tuning them, the useful thing to explore is how *sensitive* an answer is to $\alpha_L$.

## What a transport analysis reports

A transport study of a spill→capture problem typically reports:

- **A framed question** — precisely what you ask of the model, e.g. *"Does the spill reach the extraction (monitoring) well above concentration $X$, and when?"* — stated **at a receptor / along the flow axis**, not as a contaminated-area map (see the rule below).
- **A breakthrough result** — concentration vs time at the receptor well against the threshold: does it reach, exceed, and when.
- **An $\alpha_L$ sensitivity** — re-run with a different $\alpha_L$ (e.g. 10 m → 5 m) and report how arrival, peak and tail move; sensitivity stands in for calibration.
- **The defensible-claim judgment** — which threshold claims the model supports, and which it does not (below).

## ⚠️ Defensible claims: what a coarse-transverse grid supports

You meet this in [04t](04t_model_implementation.ipynb) and apply it in [08t](08t_model_application.ipynb) — it is the single most important judgment in a transport study:

| You **may** claim (grid-robust) | You may **not** claim (not grid-robust) |
|:--|:--|
| whether and roughly **how much** solute reaches the receptor well | the **lateral width** of the contaminated zone |
| **receptor arrival time** / breakthrough at the well | the **exact position** of a concentration contour |
| the **longitudinal (centreline) reach** of the plume | a **mapped contaminated area** |

**Why.** It is *not* simply that the transverse grid Péclet is large. On a grid **aligned** with the flow, MODFLOW 6's TVD scheme adds essentially **no** transverse numerical dispersion, so a high $Pe_T$ *by itself* does **not** doom the lateral width ([04t](04t_model_implementation.ipynb) verifies this directly against the 2D analytical solution). The lateral claim fails for two *other* reasons near a doublet: (a) the flow is **convergent and oblique to the grid**, which *does* produce real cross-wind numerical dispersion, and (b) the physical plume is **sub-cell** ($\alpha_T = 1$ m, narrower than even the refined ~10 m cells), so no feasible grid can *represent* it. So restrict every threshold claim to the **flow axis / receptors**; for the genuine **lateral / wellfield-protection** question, don't read it off the smeared concentration field — use **particle tracking** (a capture zone), which is advective and free of the cross-wind artefact ([08t](08t_model_application.ipynb) Rung D).

## Self-check

Before you carry these parameters into your project, make sure you understand *why* transport calibration is being handled this way.

In [ ]:
import sys, os
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')
from shared_functions import create_multiple_choice

create_multiple_choice('task_t05_checkpoint_1')

## Optional — predict, then run: how does $\alpha_L$ reshape a breakthrough? *(illustrative)*

This is a **light analytical calculation** (no MODFLOW run — it finishes in well under a second), included so you can build intuition before you run the *real* $\alpha_L$ sensitivity on the doublet in your project.

**Predict first.** For an **instantaneous pulse** released upstream and observed at a fixed distance, how does *increasing* $\alpha_L$ change the breakthrough curve — earlier or later first arrival? higher or lower peak?

We use the 1D advection–dispersion **pulse** solution `tracer_test_utils.analytical_btc`, evaluated for three dispersivities ($\alpha_L = 5, 10, 20$ m) at a receptor $x = 100$ m downstream in idealised uniform flow ($u = 1$ m/d, $\phi_e = 0.20$).

> **Units note — flux-averaged concentration (the `/u` handling).** As written, `analytical_btc` returns the pulse solution **without the $1/u$ prefactor**, so its raw output carries units of $\text{g}\,\text{m}^{-2}\,\text{d}^{-1}$, not a concentration. We therefore **divide the result by the mean water velocity $u$** in the cell below, which yields the genuine **flux-averaged** concentration in $\text{mg/L}$ ($=\text{g/m}^3$). We do the $/u$ *here* rather than editing the shared function: its only other caller fits breakthrough-curve *shape* in the legacy tracer-test utilities and is unaffected by a constant factor, so silently changing the function would need a coordinated re-check of that caller.

> **This curve is illustrative / conservative — do NOT compare it to a regulatory threshold.** It assumes idealised uniform 1D flow and an arbitrary injected mass; only its *shape* trend carries the lesson.

In [ ]:
# --- Optional alpha_L demonstration (light analytical calc; safe to leave on) ---
RUN_ALPHA_DEMO = True   # set False to skip; this is a sub-second analytical calc, not a model run

import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '../../_SUPPORT/src')
from tracer_test_utils import analytical_btc

if RUN_ALPHA_DEMO:
    # Idealised 1D uniform-flow pulse (illustrative scenario)
    u   = 1.0       # mean water velocity [m/d]
    x   = 100.0     # receptor distance [m]
    phi_e = 0.20    # locked effective porosity [-]
    M   = 1000.0    # injected mass [g] (arbitrary -> sets only the y-scale)
    A   = 10.0      # cross-sectional flow area [m^2]
    D_m = 8.64e-5   # molecular diffusion [m^2/d] (negligible here)
    t   = np.linspace(0.1, 400.0, 4000)   # time since injection [d]

    fig, ax = plt.subplots(figsize=(9, 5))
    for aL in [5.0, 10.0, 20.0]:
        D_L    = aL * u + D_m                              # longitudinal dispersion coeff [m^2/d]
        c_raw  = analytical_btc(t, x, u, D_L, M, A, phi_e)   # g/(m^2 d) -- missing the 1/u prefactor
        c_flux = c_raw / u                                 # /u -> flux-averaged mg/L (= g/m^3)
        pk = c_flux.max(); t_pk = t[c_flux.argmax()]
        ax.plot(t, c_flux, lw=2,
                label=f'alpha_L = {aL:.0f} m  (peak {pk:.2f} mg/L at {t_pk:.0f} d)')

    ax.axvline(x / u, color='0.6', ls='--', lw=1)
    ax.text(x / u, ax.get_ylim()[1] * 0.96, ' advective arrival x/u',
            color='0.4', va='top', fontsize=9)
    ax.set_xlabel('time since injection [d]')
    ax.set_ylabel('flux-averaged concentration [mg/L]')
    ax.set_title('Illustrative pulse breakthrough at x = 100 m -- effect of alpha_L\n'
                 '(conservative tracer; do NOT compare to a regulatory threshold)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('alpha_L demo skipped (RUN_ALPHA_DEMO = False).')

**What the demo shows ($\alpha_L$ key for a pulse).** Increasing $\alpha_L$ spreads the pulse more, so the breakthrough arrives **earlier** (earlier first arrival / leading toe), peaks **lower and broader**, and develops a **longer tail**. Mass is conserved, so a wider curve must be flatter — and the centre-of-mass arrival stays near the advective time $x/u$ (dashed line). This is exactly the pulse behaviour you will reproduce — on the real doublet — when you run your project's $\alpha_L$ sensitivity.

> *(For a **continuous** source the key differs: larger $\alpha_L$ gives an earlier first-arrival shoulder and a less-steep front, but the 50% breakthrough stays near $x/u$, so full plateau is reached no sooner.)*

## What you take forward

| Concept | Value |
|---|---|
| $\alpha_L$ | 10 m (locked baseline; sweep to 5 m for sensitivity) |
| $\alpha_T$ | 1 m (locked; the transverse axis is numerically under-resolved) |
| $\phi_e$ | 0.20 (locked) |
| Calibration | carried by the **flow** phase — not repeated for transport |
| Where rigour comes from | **$\alpha_L$ sensitivity** + the **defensible-claim judgment**, not parameter tuning |

**Next:** the keystone applications are in **[08t_model_application.ipynb](08t_model_application.ipynb)**.

**Navigation:** ← [04t_model_implementation.ipynb](04t_model_implementation.ipynb)  ·  → [08t_model_application.ipynb](08t_model_application.ipynb)

> *Section-completion note:* this notebook intentionally does **not** call `create_section_completion_marker` (that tracker is bound to 02t).

<details>
<summary><b>Optional — how you <i>would</i> calibrate and validate this transport model</b> (click to expand)</summary>

<br>

This model ships with **locked** transport parameters — we do **not** calibrate them here. If you ever needed to, this is the shape the exercise would take. It is a signpost, not a workflow.

**Data you'd need.** Observed **breakthrough curve(s)** — solute concentration vs. time at the monitoring / extraction well — and/or a dedicated **tracer-test BTC**. And the **flow model must be calibrated first**: transport rides on the velocity field, so heads and fluxes are pinned down before any transport parameter is touched (see [`PROJECT/flow/05f_calibration.ipynb`](../flow/05f_calibration.ipynb)).

**Parameters and their identifiability.** The transport unknowns are longitudinal dispersivity $\alpha_L$, effective porosity $\phi_e$, retardation $R$, and first-order decay $\lambda$. The catch is the **$\phi_e$–$\alpha_L$ trade-off**: *both* reshape a single breakthrough curve — a lower $\phi_e$ shifts arrival earlier (faster mean water velocity), while a larger $\alpha_L$ spreads the pulse and advances its leading toe — so one BTC alone cannot separate them. A **tracer test breaks the trade-off** by moment analysis: the mean arrival time $\bar{t}$ fixes the mean water velocity $u = x/\bar{t}$, and since $\phi_e = q/u$ (with $q$ the Darcy flux from the calibrated flow model), porosity is determined **independently**. With $\phi_e$ pinned, only $\alpha_L$ is left — a 1-D fit to the curve's spread. This is the transport analogue of the flow track's pumping-test constraint (05f): an independent measurement collapses a non-unique calibration to a single-parameter search.

**Tools & workflow.** Estimate parameters with **PEST / PEST++** — the same automated parameter-estimation engine used in [05f](../flow/05f_calibration.ipynb) — rather than manual trial-and-error, minimising an objective function such as the weighted sum of squared concentration residuals $\Phi = \sum_i w_i\,(c^{\text{sim}}_i - c^{\text{obs}}_i)^2$. **Validation is not a re-fit:** hold data back and **predict a withheld BTC or a second, independent tracer test** (split-sample), then check the blind prediction against the observations.

**Transferring from a heat-transport model.** If parameters came from a calibrated **heat**-transport twin (a common shortcut where thermal data are richer than solute data), $\alpha_L$, $\alpha_T$ and $\phi_e$ **transfer directly** — mechanical dispersion and pore space are the same for heat and solute. **Thermal retardation does _not_ transfer**: replace it with **sorption** retardation $R = 1 + \rho_b K_d / \phi_e$, whose origin is chemically specific.

**Background reading** (general concepts only — none of these endorses this specific Limmat SRC-pulse setup or its coarse-transverse simplification):
- Zheng & Bennett (2002), *Applied Contaminant Transport Modeling* (2nd ed.) — for transport-calibration concepts.
- Anderson, Woessner & Hunt (2015), *Applied Groundwater Modeling* (2nd ed.) — for groundwater-model calibration/validation framing (including split-sample validation).
- Doherty, *PEST: Model-Independent Parameter Estimation* documentation — for the PEST / PEST++ parameter-estimation workflow.

</details>

## References

\[1\] Domenico, P.A. & Schwartz, F.W. (1998). *Physical and Chemical Hydrogeology* (2nd ed.). Wiley. — analytical transport solutions (course primary text).

\[2\] Kreft, A. & Zuber, A. (1978). On the physical meaning of the dispersion equation and its solutions for different initial and boundary conditions. *Chemical Engineering Science*, 33(11), 1471–1480. — flux- vs resident-averaged concentrations.

\[3\] Langevin, C.D., et al. (2022). MODFLOW 6 Description of Input and Output. U.S. Geological Survey Techniques and Methods 6-A62. — GWT transport packages.

\[4\] Zheng, C. & Bennett, G.D. (2002). *Applied Contaminant Transport Modeling* (2nd ed.). Wiley-Interscience. — background for transport-calibration concepts (does not endorse this specific setup).

\[5\] Anderson, M.P., Woessner, W.W. & Hunt, R.J. (2015). *Applied Groundwater Modeling: Simulation of Flow and Advective Transport* (2nd ed.). Academic Press. — background for calibration / validation (split-sample) framing.

\[6\] Doherty, J. *PEST: Model-Independent Parameter Estimation — User Manual*. Watermark Numerical Computing. — background for the PEST / PEST++ parameter-estimation workflow.